In [2]:
%reload_ext autoreload
%autoreload 2

import os
import sys
from dotenv import load_dotenv
import logging

from tqdm import tqdm
import torch

from pymilvus import MilvusClient, DataType
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname('/home/hassenhamdi/Desktop/project/Hydra/HyDRA/main.py'), 'src')))

from src.utils.config_loader import ConfigLoader
from src.retrieval.engine import HyDRARetriever





In [3]:
sys.path.insert(0, '/home/hassenhamdi/Desktop/project/Hydra/HyDRA/')


In [4]:
%reload_ext autoreload
%autoreload 2
from data_processing.ingest import ingest_data
from src.services.milvus_setup import setup_milvus
from src.services.model_registry import ModelRegistry

In [5]:
client = MilvusClient(uri=os.getenv("MILVUS_URI"), token=os.getenv("MILVUS_TOKEN"))

client.drop_collection('hydra_knowledge_dev')
client.drop_collection('hydra_memory_store')
load_dotenv()
setup_milvus('development')

--- Setting up Milvus for profile: 'development' ---
--- Loading HyDRA with Deployment Profile: 'development' ---
Connecting to Milvus...
Creating knowledge collection: 'hydra_knowledge_dev'
Using dense vector data type based on profile: 102
Knowledge collection created successfully.
Creating memory collection: 'hydra_memory_store'
Memory collection created successfully.
Milvus setup complete.


In [6]:
ModelRegistry.initialize_models()


--- Initializing BGE Models (Embedding & Reranker)... ---


Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

--- Models successfully initialized on GPU (FP16). ---


In [7]:

ingest_data('/home/hassenhamdi/Desktop/project/Hydra/HyDRA/data', 'development')


Using pre-loaded BGE-M3 embedding function on GPU (FP16).
Loading documents from '/home/hassenhamdi/Desktop/project/Hydra/HyDRA/data'...


100%|██████████| 1/1 [00:00<00:00, 802.43it/s]

Loaded 1 document(s).
Splitting documents into chunks...
Created 4 chunks.
Generating dense and sparse embeddings for all chunks...



You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Embeddings generated successfully.
Preparing data for ingestion...
Ingesting 4 chunks into Milvus collection 'hydra_knowledge_dev'...


Ingesting Batches: 100%|██████████| 4/4 [00:00<00:00, 24.15it/s]

Data ingestion complete. Flushing collection to ensure data is searchable...


Collection flushed successfully.


In [8]:
def setup_logging() -> None:
    """Configure logging for better performance than print statements."""
    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(message)s',
        handlers=[logging.StreamHandler()]
    )

def check_gpu_availability() -> bool:
    """Check if CUDA GPU is available and properly initialized."""
    try:
        if not torch.cuda.is_available():
            return False
        # Test CUDA initialization
        torch.cuda.init()
        return True
    except Exception:
        return False

def main() -> None:
    """
    Optimized standalone script to test the HyDRARetriever with GPU fallback.
    """
    setup_logging()
    logger = logging.getLogger(__name__)
    
    logger.info("--- Testing HyDRARetriever ---")
    
    # Load environment variables
    load_dotenv()

    # Check GPU availability
    gpu_available = check_gpu_availability()
    if not gpu_available:
        logger.warning("CUDA GPU not available. Falling back to CPU processing.")
        # Set environment variable to force CPU mode
        os.environ['CUDA_VISIBLE_DEVICES'] = ''

    
    # Step 1: Initialize retriever with appropriate device
    try:
        retriever = HyDRARetriever(profile_name='development')
        logger.info(f"HyDRARetriever initialized successfully on {'GPU' if gpu_available else 'CPU'}.")
    except Exception as e:
        logger.error(f"Failed to initialize HyDRARetriever: {e}")
        return

    # Step 3: Define test query
    test_query = "What was the Dartmouth Workshop?"

    # Step 4: Execute query
    logger.info(f"\n--- Running test query: '{test_query}' ---")
    try:
        results = retriever.invoke(test_query)
    except Exception as e:
        logger.error(f"An error occurred during retrieval: {e}")
        return

    # Step 5: Process and display results
    logger.info("\n--- Test Results ---")
    if not results:
        logger.info("No results returned.")
        return

    # Process results efficiently
    for i, doc in enumerate(results, 1):
        logger.info(f"Result {i}:")
        logger.info(f"  Source: {doc.metadata.get('source', 'N/A')}")
        logger.info(f"  Relevance Score: {doc.metadata.get('relevance_score', 'N/A')}")
        content = doc.page_content[:300]  # Limit content length for readability
        logger.info(f"  Content: {content}...")
        logger.info("-" * 20)

    logger.info("--- Test Complete ---")


main() 
    

2025-10-07 22:15:11,368 - INFO - --- Testing HyDRARetriever ---
2025-10-07 22:15:11,376 - INFO - HyDRARetriever initialized successfully on GPU.
2025-10-07 22:15:11,376 - INFO - 
--- Running test query: 'What was the Dartmouth Workshop?' ---
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
2025-10-07 22:15:20,543 - INFO - 
--- Test Results ---
2025-10-07 22:15:20,543 - INFO - Result 1:
2025-10-07 22:15:20,544 - INFO -   Source: /home/hassenhamdi/Desktop/project/Hydra/HyDRA/data/history_of_ai.md
2025-10-07 22:15:20,544 - INFO -   Relevance Score: 0.9563904724343743
2025-10-07 22:15:20,545 - INFO -   Content: ## The Dartmouth Workshop and the Birth of AI
The term "Artificial Intelligence" was coined by John McCarthy at the Dartmouth Workshop in 1956. This event is widely considered the birthplace of AI as a 